In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from networkx.algorithms import community
import warnings

warnings.filterwarnings('ignore')

In [2]:


print("[*] KHỞI ĐỘNG HỆ THỐNG BÓC TÁCH DỮ LIỆU MẠNG LƯỚI 10 NĂM...")

# 1. LOAD MASTER PRICE MATRIX
# Bạn nhớ thay bằng file data thật của bạn
matrix_filename = "master_price_matrix.csv"
try:
    price_matrix = pd.read_csv(matrix_filename, index_col=0, parse_dates=True)
    print(f"[+] Đã load Data: {price_matrix.shape}")
except:
    print("[!] Lỗi: Không tìm thấy master_price_matrix.csv")
    exit()

years = sorted(list(set(price_matrix.index.year)))

# Chuẩn bị danh sách chứa data cho 3 bảng
macro_data = []
meso_data = []
micro_data = []

for y in years:
    print(f"  -> Đang quét X-Ray năm {y}...")
    df_year = price_matrix[price_matrix.index.year == y]
    
    # [FIX LỖI]: Bỏ đi các mã chưa lên sàn (hoặc ngừng giao dịch) trong năm đó
    # thresh=150 nghĩa là mã nào giao dịch ít nhất 150 ngày/năm thì mới lấy
    df_year = df_year.dropna(axis=1, thresh=150)
    
    # Tính lợi nhuận và loại bỏ các ngày nghỉ lễ/khuyết data
    returns = df_year.pct_change().dropna()
    
    # Nếu trong năm đó số ngày giao dịch hợp lệ < 50 ngày thì mới bỏ qua
    if len(returns) < 50:
        print(f"     [!] Năm {y} không đủ dữ liệu giao dịch, bỏ qua.")
        continue 
        
    # Tính toán Core Math
    corr = returns.corr()
    dist = np.sqrt(2 * (1 - corr))
    
    # ... (Phần code dựng Graph G và tính toán ở dưới giữ nguyên không đổi) ...
    
    G = nx.Graph()
    stocks = dist.columns
    for i in range(len(stocks)):
        for j in range(i + 1, len(stocks)):
            w = dist.iloc[i, j]
            if not np.isnan(w):
                G.add_edge(stocks[i], stocks[j], weight=w)
                
    mst = nx.minimum_spanning_tree(G)
    
    # --- TÍNH TOÁN CÁC CHỈ SỐ ---
    centrality = nx.degree_centrality(mst) # Độ quyền lực (số kết nối)
    betweenness = nx.betweenness_centrality(mst) # Độ "Trạm trung chuyển"
    communities = list(community.greedy_modularity_communities(mst))
    
    total_mst_length = sum([d['weight'] for u, v, d in mst.edges(data=True)])
    avg_return = returns.mean().mean() * 252 # Lợi nhuận trung bình năm hóa
    
    king_of_year = max(centrality, key=centrality.get)
    
    # 1. GHI VÀO MACRO (VĨ MÔ)
    macro_data.append({
        'Year': y,
        'MST_Length': round(total_mst_length, 2),
        'Num_Communities': len(communities),
        'Market_King': king_of_year,
        'Avg_Market_Return': round(avg_return, 4)
    })
    
    # 2. GHI VÀO MESO (TRUNG MÔ - BĂNG ĐẢNG)
    for c_id, comm in enumerate(communities):
        members = list(comm)
        # Tìm thủ lĩnh của băng đảng này
        comm_leader = max(members, key=lambda x: centrality.get(x, 0))
        
        meso_data.append({
            'Year': y,
            'Community_ID': f"C_{y}_{c_id}",
            'Leader': comm_leader,
            'Size': len(members),
            'Members': ", ".join(members)
        })
        
        # 3. GHI VÀO MICRO (VI MÔ - TỪNG CỔ PHIẾU)
        # Sắp xếp xếp hạng quyền lực của toàn bộ 50 mã trong năm
        sorted_stocks_by_power = sorted(centrality, key=centrality.get, reverse=True)
        
        for stock in members:
            micro_data.append({
                'Year': y,
                'Symbol': stock,
                'Community_Leader': comm_leader,
                'Power_Score': round(centrality[stock], 4),
                'Bridge_Score': round(betweenness[stock], 4), # Quan trọng để xem mã nào nối 2 nhóm với nhau
                'Power_Rank': sorted_stocks_by_power.index(stock) + 1,
                'Yearly_Return': round(returns[stock].mean() * 252, 4)
            })

# Convert sang DataFrames
df_macro = pd.DataFrame(macro_data)
df_meso = pd.DataFrame(meso_data)
df_micro = pd.DataFrame(micro_data)

# Lưu Database
df_macro.to_csv("db_macro_network.csv", index=False)
df_meso.to_csv("db_meso_community.csv", index=False)
df_micro.to_csv("db_micro_nodes.csv", index=False)

print("\n[+] HOÀN TẤT! Đã đóng gói 3 bảng dữ liệu Database.")

[*] KHỞI ĐỘNG HỆ THỐNG BÓC TÁCH DỮ LIỆU MẠNG LƯỚI 10 NĂM...
[+] Đã load Data: (4997, 50)
  -> Đang quét X-Ray năm 2006...
  -> Đang quét X-Ray năm 2007...
  -> Đang quét X-Ray năm 2008...
  -> Đang quét X-Ray năm 2009...
  -> Đang quét X-Ray năm 2010...
  -> Đang quét X-Ray năm 2011...
  -> Đang quét X-Ray năm 2012...
  -> Đang quét X-Ray năm 2013...
  -> Đang quét X-Ray năm 2014...
  -> Đang quét X-Ray năm 2015...
  -> Đang quét X-Ray năm 2016...
  -> Đang quét X-Ray năm 2017...
  -> Đang quét X-Ray năm 2018...
  -> Đang quét X-Ray năm 2019...
  -> Đang quét X-Ray năm 2020...
  -> Đang quét X-Ray năm 2021...
  -> Đang quét X-Ray năm 2022...
  -> Đang quét X-Ray năm 2023...
  -> Đang quét X-Ray năm 2024...
  -> Đang quét X-Ray năm 2025...
  -> Đang quét X-Ray năm 2026...
     [!] Năm 2026 không đủ dữ liệu giao dịch, bỏ qua.

[+] HOÀN TẤT! Đã đóng gói 3 bảng dữ liệu Database.
